In [1]:
import time
import sys
from pathlib import Path

import numpy as np
import scipy.linalg as la

import warp as wp

# insert parent directory in path so we can import socu without installing it
sys.path.insert(0, str(Path().resolve().parents[0]))
from socu.block_tridiag_solver import *

In [2]:
wp.init()

Warp 1.11.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 3080" (10 GiB, sm_86, mempool enabled)
   Kernel cache:
     /home/rschwan/.cache/warp/1.11.0


In [3]:
def generate_random_psd_block_tridiag(n: int, N: int):
    np.random.seed(0)

    L = np.zeros((N * n, N * n))
    for i in range(N):
        L[i*n:(i+1)*n, i*n:(i+1)*n] = np.tril(np.random.rand(n, n)) + 10 * np.eye(n)
        if i < N - 1:
            L[(i+1)*n:(i+2)*n, i*n:(i+1)*n] = np.random.rand(n, n)

    A = L @ L.T

    return L, A

In [4]:
n = 34
N = 128
block_dim = 64
block_size = 32
dtype = wp.float64
_, A = generate_random_psd_block_tridiag(n, N)

In [5]:

L_np = np.zeros((N, n, n), dtype=np.float64)
for i in range(N):
    L_np[i, :, :] = A[i*n:(i+1)*n, i*n:(i+1)*n]

n_E = calculate_off_diag_storage_len(N)
E_np = np.zeros((n_E, n, n), dtype=np.float64)
for i in range(N - 1):
    E_np[i, :, :] = A[(i+1)*n:(i+2)*n, i*n:(i+1)*n]

In [6]:
np.random.seed(0)
b_np = np.random.rand(N * n)
Lc_np = la.cholesky(A, lower=True)
#x_np = la.solve(A, b_np)
y_np = la.solve_triangular(Lc_np, b_np, lower=True)
x_np = la.solve_triangular(Lc_np.T, y_np, lower=False)
print(f'error: {la.norm(b_np - A @ x_np)}')

error: 1.094314663144768e-14


In [7]:
L = wp.from_numpy(L_np, dtype=dtype, device='cuda')
E = wp.from_numpy(E_np, dtype=dtype, device='cuda')
x = wp.from_numpy(b_np.reshape(N, n, 1), dtype=dtype, device='cuda')

In [8]:
use_cuda_graph = False
cholesky_factor_launch = create_cholesky_factor_launch(L, E, device='cuda', block_dim=block_dim, block_size=block_size, use_cuda_graph=use_cuda_graph, dtype=dtype)
cholesky_solve_launch = create_cholesky_solve_launch(L, E, x, device='cuda', block_dim=block_dim, block_size=block_size, use_cuda_graph=use_cuda_graph, dtype=dtype)
cholesky_factor_and_solve_launch = create_cholesky_factor_and_solve_launch(L, E, x, device='cuda', block_size=block_size, block_dim=block_dim, use_cuda_graph=use_cuda_graph, dtype=dtype)

Module cholesky_factor_potrf_l_blocked_kernel e0fca86 load on device 'cuda:0' took 210.11 ms  (cached)
Module cholesky_factor_trsm_rltn_blocked_kernel 5bfac50 load on device 'cuda:0' took 238.83 ms  (cached)
Module cholesky_factor_trsm_llnn_blocked_kernel 9e32036 load on device 'cuda:0' took 250.36 ms  (cached)
Module cholesky_factor_syrk_ln_blocked_kernel 908a3cf load on device 'cuda:0' took 231.33 ms  (cached)
Module cholesky_factor_syrk_lt_blocked_kernel cbfa609 load on device 'cuda:0' took 231.87 ms  (cached)
Module cholesky_factor_gemm_nn_blocked_kernel 99aed01 load on device 'cuda:0' took 296.25 ms  (cached)
Module forward_substitution_trsm_llnn_blocked_kernel 1a728aa load on device 'cuda:0' took 136.71 ms  (cached)
Module forward_substitution_gemm_nn_blocked_kernel b992c57 load on device 'cuda:0' took 151.82 ms  (cached)
Module forward_substitution_gemm_tn_blocked_kernel fb0eab2 load on device 'cuda:0' took 149.95 ms  (cached)
Module backward_substitution_gemm_tn_blocked_kernel 

In [9]:
wp.copy(L, wp.from_numpy(L_np, dtype=dtype))
wp.copy(E, wp.from_numpy(E_np, dtype=dtype))
wp.copy(x, wp.from_numpy(b_np.reshape(N, n, 1), dtype=dtype))
wp.synchronize()

s = time.time()
cholesky_factor_launch()
wp.synchronize()
e = time.time()
print(f'cholesky_factor: {(e-s)*1e3} ms')

s = time.time()
cholesky_solve_launch()
wp.synchronize()
e = time.time()
print(f'cholesky_solve: {(e-s)*1e3} ms')

print(f'x error: {la.norm(x_np - x.numpy().flatten())}')
print(f'b error: {la.norm(b_np - A @ x.numpy().flatten())}')

cholesky_factor: 1.178741455078125 ms
cholesky_solve: 0.8401870727539062 ms
x error: 1.4736650102099992e-16
b error: 1.2801368354234253e-14


In [10]:
wp.copy(L, wp.from_numpy(L_np, dtype=dtype))
wp.copy(E, wp.from_numpy(E_np, dtype=dtype))
wp.copy(x, wp.from_numpy(b_np.reshape(N, n, 1), dtype=dtype))
wp.synchronize()

s = time.time()
cholesky_factor_and_solve_launch()
wp.synchronize()
e = time.time()
print(f'cholesky_factor_and_solve: {(e-s)*1e3} ms')

print(f'x error: {la.norm(x_np - x.numpy().flatten())}')
print(f'b error: {la.norm(b_np - A @ x.numpy().flatten())}')

cholesky_factor_and_solve: 1.306772232055664 ms
x error: 1.4709252389199684e-16
b error: 1.2903940405275627e-14


In [11]:
use_cuda_graph = True
cholesky_factor_launch = create_cholesky_factor_launch(L, E, device='cuda', block_dim=block_dim, block_size=block_size, use_cuda_graph=use_cuda_graph, dtype=dtype)
cholesky_solve_launch = create_cholesky_solve_launch(L, E, x, device='cuda', block_dim=block_dim, block_size=block_size, use_cuda_graph=use_cuda_graph, dtype=dtype)
cholesky_factor_and_solve_launch = create_cholesky_factor_and_solve_launch(L, E, x, device='cuda', block_dim=block_dim, block_size=block_size, use_cuda_graph=use_cuda_graph, dtype=dtype)

In [12]:
wp.copy(L, wp.from_numpy(L_np, dtype=dtype))
wp.copy(E, wp.from_numpy(E_np, dtype=dtype))
wp.copy(x, wp.from_numpy(b_np.reshape(N, n, 1), dtype=dtype))
wp.synchronize()

s = time.time()
cholesky_factor_launch()
wp.synchronize()
e = time.time()
print(f'cholesky_factor: {(e-s)*1e3} ms')

s = time.time()
cholesky_solve_launch()
wp.synchronize()
e = time.time()
print(f'cholesky_solve: {(e-s)*1e3} ms')

print(f'x error: {la.norm(x_np - x.numpy().flatten())}')
print(f'b error: {la.norm(b_np - A @ x.numpy().flatten())}')

cholesky_factor: 0.8180141448974609 ms
cholesky_solve: 0.4496574401855469 ms
x error: 1.463646346263172e-16
b error: 1.2826327190061371e-14


In [13]:
wp.copy(L, wp.from_numpy(L_np, dtype=dtype))
wp.copy(E, wp.from_numpy(E_np, dtype=dtype))
wp.copy(x, wp.from_numpy(b_np.reshape(N, n, 1), dtype=dtype))
wp.synchronize()

s = time.time()
cholesky_factor_and_solve_launch()
wp.synchronize()
e = time.time()
print(f'cholesky_factor_and_solve: {(e-s)*1e3} ms')

print(f'x error: {la.norm(x_np - x.numpy().flatten())}')
print(f'b error: {la.norm(b_np - A @ x.numpy().flatten())}')

cholesky_factor_and_solve: 1.0521411895751953 ms
x error: 1.4632714529302505e-16
b error: 1.2821488489084104e-14


In [14]:
import jax
import jax.numpy as jnp
from jax import config, jit
from socu.jax import cholesky_factor, cholesky_solve

config.update("jax_enable_x64", True)

L_jax = jnp.array(L_np, dtype=jnp.float64)
E_jax = jnp.array(E_np, dtype=jnp.float64)
b_jax = jnp.array(b_np, dtype=jnp.float64).reshape((N, -1, 1))

cholesky_factor(L_jax, E_jax) # warm-start
s = time.time()
L_factor, E_factor = cholesky_factor(L_jax, E_jax)
e = time.time()
print(f'cholesky_factor: {(e-s)*1e3} ms')

cholesky_solve(L_factor, E_factor, b_jax) # warm-start
s = time.time()
x_jax = cholesky_solve(L_factor, E_factor, b_jax)
e = time.time()
print(f'cholesky_solve: {(e-s)*1e3} ms')

print(f'x error: {la.norm(x_np - np.array(x_jax).flatten())}')
print(f'b error: {la.norm(b_np - A @ np.array(x_jax).flatten())}')

Module socu.jax edde159 load on device 'cuda:0' took 0.38 ms  (cached)
Module cholesky_factor_potrf_l_blocked_kernel 8df96c7 load on device 'cuda:0' took 231.69 ms  (cached)
Module cholesky_factor_trsm_rltn_blocked_kernel 72ddce5 load on device 'cuda:0' took 270.97 ms  (cached)
Module cholesky_factor_trsm_llnn_blocked_kernel fe723d8 load on device 'cuda:0' took 277.89 ms  (cached)
Module cholesky_factor_syrk_ln_blocked_kernel ecba793 load on device 'cuda:0' took 268.81 ms  (cached)
Module cholesky_factor_syrk_lt_blocked_kernel 159adc2 load on device 'cuda:0' took 263.99 ms  (cached)
Module cholesky_factor_gemm_nn_blocked_kernel 8d8abe3 load on device 'cuda:0' took 354.86 ms  (cached)
cholesky_factor: 15.526294708251953 ms
Module forward_substitution_trsm_llnn_blocked_kernel 04a0016 load on device 'cuda:0' took 146.48 ms  (cached)
Module forward_substitution_gemm_nn_blocked_kernel e6ba7ca load on device 'cuda:0' took 163.29 ms  (cached)
Module forward_substitution_gemm_tn_blocked_kernel

In [15]:
cholesky_factor_jit = jit(cholesky_factor)
cholesky_solve_jit = jit(cholesky_solve)

cholesky_factor_jit(L_jax, E_jax) # warm-start
s = time.time()
L_factor, E_factor = cholesky_factor_jit(L_jax, E_jax)
e = time.time()
print(f'cholesky_factor: {(e-s)*1e3} ms')

cholesky_solve_jit(L_factor, E_factor, b_jax) # warm-start
s = time.time()
x_jax = cholesky_solve_jit(L_factor, E_factor, b_jax)
e = time.time()
print(f'cholesky_solve: {(e-s)*1e3} ms')

print(f'x error: {la.norm(x_np - np.array(x_jax).flatten())}')
print(f'b error: {la.norm(b_np - A @ np.array(x_jax).flatten())}')

cholesky_factor: 0.11849403381347656 ms
cholesky_solve: 0.7271766662597656 ms
x error: 1.3581224512947034e-16
b error: 1.0943481565215685e-14
